In [3]:
import json
import re
import pandas as pd


pf = pd.read_json("../data/clean/law.json", lines=True, nrows=10)
records = pf.to_dict("records")

def extract_legal_metadata(content):
    first_line = content.split('\n')[0].strip()

    match = re.search(r'(Điều\s+\d+[a-zA-Z]*)\.\s*(.*)', first_line)

    if match:
        article_number = match.group(1).strip()
        section_title = match.group(2).strip()
        return article_number, section_title

    return "Không xác định", "Không xác định"

output_file = "../data/chunk/law-test.json"

with open(output_file, 'w', encoding='utf-8') as f:
    for index, item in enumerate(records):

        text_content = item["content"]
        source_name = item["source"]

        article_number, section_title = extract_legal_metadata(text_content)

        safe_source_name = source_name.lower().replace(" ", "_")
        chunk_id = f"{safe_source_name}_{index:04d}"

        chunk_data = {
            "chunk_id": chunk_id,
            "text": text_content,
            "metadata": {
                "source": source_name,
                "category": "Luat_Phap",
                "section_title": section_title,
                "article_number": article_number,
                "chunk_index": index
            }
        }

        f.write(json.dumps(chunk_data, ensure_ascii=False) + "\n")

print("✅ Đã xử lý và đóng gói thành công!")

ValueError: Expected object or value

In [ ]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
)

#pf = pd.read_csv("../data/clean/kinh-te-chinh-tri.csv")
pf = pd.read_csv("ktct.csv")
records = pf.to_dict("records")

output_file = "ktct.jsonl"

with open(output_file, "w", encoding="utf-8") as f:

    global_chunk_counter = 0

    for doc_index, item in enumerate(records):

        raw_text = item["content"]
        source_name = item["source"]

        category = (
            "Giao_Trinh"
            if "Giao Trinh" in source_name
            else "Wikipedia"
        )

        chunks = text_splitter.split_text(raw_text)

        for internal_index, chunk_text in enumerate(chunks):

            safe_source_name = (
                source_name.lower()
                .replace(".pdf", "")
                .replace(" ", "_")
            )

            chunk_id = (
                f"{safe_source_name}"
                f"_doc{doc_index:03d}"
                f"_chunk{internal_index:03d}"
            )

            chunk_data = {
                "chunk_id": chunk_id,
                "text": chunk_text,
                "metadata": {
                    "source": source_name,
                    "category": category,
                    "section_title": item["title"],
                    "page_number": item["page"],
                    "chunk_index": internal_index
                }
            }

            f.write(
                json.dumps(chunk_data, ensure_ascii=False)
                + "\n"
            )

            global_chunk_counter += 1

print(
    f"✅ Đã tạo {global_chunk_counter} chunks"
)

In [ ]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Gọi Tokenizer của bge-m3
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [ ]:
import json
import pandas as pd
from tqdm.auto import tqdm
# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "../data/clean/vi-wiki.csv"
OUTPUT_FILE = "../data/chunk/vi-wiki.jsonl"

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 400

text_splitter = (
    RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
)


# ============================================================
# LOAD DATA
# ============================================================

pf = pd.read_csv(INPUT_FILE)

total_chunks = 0


# ============================================================
# CHUNKING
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for doc_index, row in tqdm(
        pf.iterrows(),
        total=len(pf),
        desc="Chunking Wiki"
    ):

        text = str(row.get("text", "")).strip()

        if not text:
            continue

        title = str(row.get("title", "")).strip()

        chunks = text_splitter.split_text(text)

        for chunk_index, chunk_text in enumerate(chunks):

            chunk_data = {
                "chunk_id": (
                    f"wiki_"
                    f"doc{doc_index:06d}_"
                    f"chunk{chunk_index:03d}"
                ),
                "text": chunk_text,
                "metadata": {
                    "source": "vi-wiki",
                    "category": "Wikipedia",
                    "title": title,
                    "doc_index": int(doc_index),
                    "chunk_index": int(chunk_index)
                }
            }

            f.write(
                json.dumps(
                    chunk_data,
                    ensure_ascii=False
                )
                + "\n"
            )

            total_chunks += 1


print(
    f"\n✅ Hoàn thành:"
    f"\n   Documents: {len(pf):,}"
    f"\n   Chunks: {total_chunks:,}"
    f"\n   Output: {OUTPUT_FILE}"
)

In [1]:
import os

import json
import glob
from tqdm import tqdm
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

def build_chroma_db(jsonl_folder, chroma_save_dir, batch_size=300):
    print("🚀 Load BAAI/bge-m3 on GPU (Parallelism Disabled)...")

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True, 'batch_size': 16}
    )

    print(f"🗄️ Init ChromaDB: {chroma_save_dir}")

    vectorstore = Chroma(
        collection_name="rag_vietnamese_docs",
        embedding_function=embeddings,
        persist_directory=chroma_save_dir
    )

    documents_batch = []
    total_processed = 0

    files = glob.glob(f"{jsonl_folder}/law1.jsonl")
    
    for file_path in files:
        print(f"\n📄 Processing file: {file_path}")

        total_lines = 0
        with open(file_path, 'r', encoding='utf-8') as f:
            total_lines = sum(1 for _ in f)

        file_bar = tqdm(total=total_lines, desc="Tiến trình File", leave=True)

        with open(file_path, 'r', encoding='utf-8') as f:
            for line_idx, line in enumerate(f):
                data = json.loads(line)
                meta = data.get("metadata", {})

                source = meta.get("source", "") #Bộ Luật dân sự
                article_number = meta.get("article_number", "") #Điều 12
                section_title = meta.get("section_title", "") #Kinh tế thị trường
                full_title = f"{source} {article_number}. {section_title}"
                page_content = f"Tiêu đề: {full_title}\n\nNội dung:\n{data.get('text', '')}"

                doc = Document(
                    page_content=page_content,
                    metadata={**meta, "full_title": full_title}
                )
                documents_batch.append(doc)
                file_bar.update(1)

                if len(documents_batch) >= batch_size:
                    
                    try:
                        vectorstore.add_documents(documents_batch)
                                                
                        total_processed += len(documents_batch)
                        documents_batch = [] 
                        
                    except Exception as e:
                        print(f"\n❌ LỖI NGHIÊM TRỌNG TẠI DÒNG {line_idx}!")
                        print(f"Nội dung lỗi: {e}")
                        print(f"Text gây lỗi: {documents_batch[-1].page_content[:200]}")
                        return

        file_bar.close()

    if documents_batch:
        vectorstore.add_documents(documents_batch)
        total_processed += len(documents_batch)

    print(f"\n✅ Done! Total indexed chunks: {total_processed}")

if __name__ == "__main__":
    FILE_JSONL = "../data/chunk/" 
    THU_MUC_LUU_CHROMA = "./my_chroma_db"
    
    build_chroma_db(FILE_JSONL, THU_MUC_LUU_CHROMA, batch_size=32)

🚀 Load BAAI/bge-m3 on GPU (Parallelism Disabled)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

🗄️ Init ChromaDB: ./my_chroma_db

📄 Processing file: ../data/chunk//law1.jsonl


Tiến trình File: 100%|███████████████████████████████████████████████████████████| 22996/22996 [40:34<00:00,  9.44it/s]



✅ Done! Total indexed chunks: 22996


In [1]:
# import chromadb

# client = chromadb.PersistentClient(path="./my_chroma_db")

# client.delete_collection("rag_vietnamese_docs")
# print("🗑️ Deleted collection")



🗑️ Deleted collection


In [3]:
query = "Điều kiện để giao kết hợp đồng dân sự là gì?"

docs = vectorstore.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(docs, 1):
    print(f"\n===== Result {i} =====")
    print(doc.page_content[:1000])

NameError: name 'vectorstore' is not defined